# Module 21 — Observability and Tracing

Companion notebook to [`docs/modules/21_observability_and_tracing.md`](../docs/modules/21_observability_and_tracing.md).

Genuinely new code, not a rebuild — `packages/local_ai_core/tracing/` was scaffolded empty
back in Phase 0. Every piece here is real, deterministic, model-free: structured JSON logs,
PII redaction, a metrics registry matching curriculum's exact metric table, a real trace-span
tree, and a SQLite-backed evaluation/feedback store.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_21"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. Labs 1-2: structured logs and request IDs

In [2]:
from structured_logs_demo import run_lab as run_logs_lab, result_to_markdown as logs_to_markdown

logs_result = run_logs_lab()
print(logs_to_markdown(logs_result))

# Labs 1-2 - structured logs and request IDs

- Request ID (Module 6's `ensure_trace_id()`): `667887e1-d7c0-4043-89a4-beb7ee1b814d`

| Policy | Logged fields |
|---|---|
| full | `{"prompt": "Classify this ticket for jane.doe@example.com: I was charged twice.", "prompt_logging_policy": "full"}` |
| redacted | `{"prompt": "Classify this ticket for [EMAIL]: I was charged twice.", "prompt_logging_policy": "redacted"}` |
| hash_only | `{"prompt_hash": "7fceb095ee8cbdd9", "prompt_logging_policy": "hash_only"}` |
| none | `{"prompt_logging_policy": "none"}` |



## 2. Lab 3: trace spans — the full curriculum trace model

In [3]:
from trace_spans_demo import run_lab as run_trace_lab, result_to_markdown as trace_to_markdown

trace_result = run_trace_lab()
print(trace_to_markdown(trace_result))

# Lab 3 - trace spans

- Request ID: req-demo-001
- Span order: input_validation -> prompt_template_version -> retrieval_query -> retrieved_chunk_ids -> reranker_scores -> context_packing -> model_call -> output_validation -> final_response -> evaluation_hooks
- Total elapsed: 3.79ms
- Missing core steps: []



## 3. Lab 4: trace RAG retrieval — a real vector search

In [4]:
from rag_retrieval_trace_demo import run_lab as run_rag_lab, result_to_markdown as rag_to_markdown

rag_result = run_rag_lab()
print(rag_to_markdown(rag_result))

# Lab 4 - trace RAG retrieval

- Retrieved doc IDs: ['password-reset-guide', 'billing-faq']
- Top score: 0.9939
- Trace spans recorded: ['retrieval_query', 'retrieved_chunk_ids', 'reranker_scores']



## 4. Lab 5: trace tool calls

In [5]:
from tool_call_trace_demo import run_lab as run_tool_lab, result_to_markdown as tool_to_markdown

tool_result = run_tool_lab()
print(tool_to_markdown(tool_result))

# Lab 5 - trace tool calls

- lookup_order({'order_id': 'ORD-123'}) -> ok
- cancel_order({'order_id': 'ORD-999'}) -> error

Metrics: tool_call_count=2, tool_error_count=1



## 5. PII redaction — real regex-based detection

In [6]:
from local_ai_core.tracing.pii_redaction import redact_pii

sample = "Contact jane.doe@example.com or call 555-867-5309. SSN on file: 123-45-6789."
redaction_result = redact_pii(sample)
print(f"Redacted text: {redaction_result.redacted_text}")
print(f"Redaction counts: {redaction_result.redaction_counts}")
print(f"Total redactions: {redaction_result.total_redactions}")

Redacted text: Contact [EMAIL] or call [PHONE]. SSN on file: [SSN].
Redaction counts: {'email': 1, 'ssn': 1, 'phone': 1}
Total redactions: 3


## 6. Lab 6: local observability dashboard/report

In [7]:
from observability_dashboard_demo import run_lab as run_dashboard_lab, result_to_markdown as dashboard_to_markdown

dashboard_result = run_dashboard_lab()
print(dashboard_to_markdown(dashboard_result))

# Lab 6 - observability dashboard

- Requests traced: 3
- Mean trace latency: 0.0021ms
- Feedback summary: {'down': 1, 'up': 2}
- Traces missing a core step: []



## Summary

- Structured logs: real JSON log lines under all four prompt-logging policies, proving
  HASH_ONLY/NONE never leak the raw prompt.
- Trace spans: a complete, correctly-ordered trace matching curriculum's exact trace model,
  with real measured elapsed time.
- RAG retrieval trace: a real vector search recorded into the trace, correct nearest document
  first.
- Tool call trace: a real success and a real failure, trace and metrics agreeing on counts.
- PII redaction: real regex-based detection across four categories with no double-counting.
- Observability dashboard: metrics, traces, eval scores, and user feedback tied together by
  trace_id into one report.